# Cell2location deconvolution using three reference strategies

* This notebook compares Cell2Location deconvolution results across three sc-RNA-seq reference-building strategies tested during the `spatial-sc-rnaseq-integration-wf` development.
* Inputs are the outputs of `reference_building/benchmark/reference_data/ma/scripts/03_build_benchmark.py`, `06_submit_cell2location_RefStrategy*sh` and `07_agreement_matrix_reference_RefStrategy*.py`

| Strategy | sc-rnaseq-wf | ASAP cells | Siletti cells | Kamath cells | Integration logic | Labels used for training |
|---|---|---|---|---|---|---|
| **1** | v3.1.0 | ~100k random (incl. 20% that passed `sc-rnaseq-wf` v3.1.0 QC but were filtered out in v5.0.0) | ~100k (all ~805 DA VGLUT2 + random top-up from all cell types) | ~100k, **all cell types** (200-cell-floor-per-type + random top-up; incl. ~6.5k out of ~22k Kamath's DAs |   Repo's own `integrate_kamath_menon_siletti_asap.py` — scVI + scANVI trained from scratch | Kamath labels (ground truth for scANVI) |
| **2** | v5.0.0 | 200k random | 100k random | ~22k, **DA-only** (no other cell types — `sc-rnaseq-wf` only spikes in Kamath's DA population) | `sc-rnaseq-wf`'s own pipeline — scVI/MMC/scANVI | `sc-rnaseq-wf` used Siletti/Allen labels for scANVI (`C_scANVI`), reused as-is |
| **3** | v5.0.0 | 100k (~68k remapped from Strategy 1 + ~11k extra labeled as DA by `sc-rnaseq-wf` v5.0.0 + ~21k random fill) | Same as Strategy 1 | Same as Strategy 1 | No retraining — reuses `sc-rnaseq-wf` v5.0.0 | `sc-rnaseq-wf` used Siletti/Allen labels, but different than Strategy 2. For ASAP cells, used `sc-rnaseq-wf` v5.0.0 Siletti/Allen `C_scANVI`. For Siletti cells, used the original Siletti/Allen labels. For Kamath cells, relabeled into Siletti/Allen space via `utils/cell_type_maps.py`'s `KAMATH_TO_SILETTI_ALLEN`


See `reference_building/benchmark/README.md` for details.


## Imports

In [ ]:
import glob
import sys
import functools
import warnings
from pathlib import Path

from IPython.display import Image, HTML, display
import ipywidgets as widgets

import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import asyncio

# Needed in this oder to work in repo promoted and untracked notebook
repo_name = "spatial-sc-rnaseq-integration-wf"
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from notebook_utils import get_repo_root # must come after the utils sys.path.insert above
repo_root = get_repo_root(repo_name)

sys.path.insert(0, str(repo_root / "utils"))
from genomics_constants import KAMATH_DA_MARKER_SYMBOLS
from input_definitions import DECONVOLUTED_H5AD_SUFFIX, MA_STRATEGIES, ma_base_dir, ma_strategy_dirname
from notebook_utils import (
    build_all_vs_all_figure,
    coarse_category as uber_type_of,
    dropdown,
    inject_big_dropdown_css,
    load_all_vs_all,
    load_kamath_collapse_and_palette,
    selected_cell_types,
    spatial_scatter_grid,
    strip_q05_abundance_prefix,
)

pio.renderers.default = "plotly_mimetype"
sc.set_figure_params(dpi=100, frameon=False)
warnings.filterwarnings("ignore", message=".*deprecated.*")

## Inputs

In [ ]:
BASE_INDIR = str(ma_base_dir(repo_root))

STRATEGIES = MA_STRATEGIES
STRATEGY_DIR = {n: ma_strategy_dirname(n) for n in STRATEGIES}
H5AD_SUFFIX = DECONVOLUTED_H5AD_SUFFIX
BASE_STRATEGY = 1

available_samples = sorted(
    Path(p).name.removesuffix(H5AD_SUFFIX)
    for p in glob.glob(str(Path(
        BASE_INDIR, STRATEGY_DIR[BASE_STRATEGY], f"*{H5AD_SUFFIX}"
    )))
)
print(f"Found {len(available_samples)} sample(s) in {BASE_INDIR}: {available_samples}")

## Utils

In [ ]:
all_vs_all_by_strategy, row_counts_by_strategy, col_counts_by_strategy = {}, {}, {}
for n in STRATEGIES:
    all_vs_all_by_strategy[n], row_counts_by_strategy[n], col_counts_by_strategy[n] = load_all_vs_all(
        Path(BASE_INDIR, STRATEGY_DIR[n])
    )


SUBTITLE_BY_STRATEGY = {
    1: "Strategy 1",
    2: "Strategy 2",
    3: "Strategy 3",
}

_sort_by_dropdown_compare = dropdown(
    big=True,
    options=["counts (max to min)", "alphanumeric", "cluster"],
    description="Sort by:",
    layout=widgets.Layout(width="420px", height="56px"),
)
_strategy_outs = {n: widgets.Output() for n in STRATEGIES}

# Agreement plots
def _redraw_all_vs_all_compare(_=None):
    sort_by = _sort_by_dropdown_compare.value
    for n in STRATEGIES:
        _strategy_outs[n].clear_output(wait=True)
        with _strategy_outs[n]:
            display(build_all_vs_all_figure(
                all_vs_all_by_strategy[n], row_counts_by_strategy[n], col_counts_by_strategy[n],
                sort_by, f"{SUBTITLE_BY_STRATEGY[n]} (10 Ma samples pooled)",
                wide_factor=0.75, height_factor=0.75,
                y_annotation_offset=-0.45, x_annotation_offset=-0.48,
            ))


# Spatial plots
def _redraw_spatial_compare(_=None):
    column = cell_type_dropdown.value
    _spatial_compare_out.clear_output(wait=True)
    with _spatial_compare_out:
        fig, axs = plt.subplots(1, len(STRATEGIES), figsize=(4 * len(STRATEGIES), 4))
        for ax, n in zip(axs, STRATEGIES):
            strategy_adata = _adata_by_strategy[n]
            vmin, vmax = _vrange_widgets[n].value
            sq.pl.spatial_scatter(
                strategy_adata, color=column, cmap="magma",
                vmin=vmin, vmax=vmax, size=1.3,
                img_res_key="hires", title=f"Strategy {n}\n{column}", ax=ax,
            )
            # Crop to the tissue
            library_id = list(strategy_adata.uns["spatial"].keys())[0]
            scalef = strategy_adata.uns["spatial"][library_id]["scalefactors"]["tissue_hires_scalef"]
            coords = strategy_adata.obsm["spatial"] * scalef
            (x_min, y_min), (x_max, y_max) = coords.min(axis=0), coords.max(axis=0)
            x_pad, y_pad = (x_max - x_min) * 0.05, (y_max - y_min) * 0.05
            ax.set_xlim(x_min - x_pad, x_max + x_pad)
            ax.set_ylim(y_max + y_pad, y_min - y_pad)
        plt.tight_layout()
        display(fig)
        plt.close(fig)


def _reset_vmin_vmax(column):
    for n in STRATEGIES:
        values = _adata_by_strategy[n].obs[column].values
        data_max = float(np.nanmax(values)) or 1.0
        default_vmax = float(np.nanpercentile(values, _DEFAULT_VMAX_PERCENTILE))
        slider = _vrange_widgets[n]
        with slider.hold_trait_notifications():
            slider.min = 0.0
            slider.max = data_max
            slider.step = data_max / 200
            slider.value = (0.0, default_vmax)
        _update_slider_label(n)


def _reset_vmin_vmax_and_redraw(_=None):
    _reset_vmin_vmax(cell_type_dropdown.value)
    _redraw_spatial_compare()


# Scatter plot
def _load_reference_abundance(source, sample_name):
    """
    Load one reference's per-spot cell-type abundance for `sample_name`.

    Parameters
    ----------
    source : str
        Key into `_REFERENCE_OPTIONS` (e.g. "Strategy 1").
    sample_name : str
        Sample ID, matching `sample_dropdown`'s options.

    Returns
    -------
    pandas.DataFrame
        Spots x cell types (short names), from obsm["q05_cell_abundance_w_sf"].
    """
    n = _REFERENCE_OPTIONS[source]
    path = Path(BASE_INDIR, STRATEGY_DIR[n], sample_name + H5AD_SUFFIX)
    abund = ad.read_h5ad(path).obsm["q05_cell_abundance_w_sf"].copy()
    abund.columns = [strip_q05_abundance_prefix(c) for c in abund.columns]
    return abund


def _rgb_str(rgb):
    """Convert an (r, g, b) float-in-[0,1] tuple to a plotly "rgb(...)" string."""
    return f"rgb({int(rgb[0]*255)},{int(rgb[1]*255)},{int(rgb[2]*255)})"


# Sliders
def _fmt_bound(value):
    return "0" if value == 0 else f"{value:.3f}"

    
def _update_slider_label(n, _=None):
    vmin, vmax = _vrange_widgets[n].value
    _vrange_widgets[n].description = f"Strategy {n} ({_fmt_bound(vmin)} - {_fmt_bound(vmax)}):"

# Dropdown menus
inject_big_dropdown_css()

DA_UBER_CATEGORIES = {"Dopaminergic", "CALB1", "SOX6"}

sample_dropdown = dropdown(big=True, options=available_samples, description="Sample:")

# Used by selected_cell_types(), called once per strategy in section 2 to restrict to CALB1_*/SOX6_* cell types.
cell_type_view_dropdown = dropdown(
    big=True, options=["All cell types", "DAs only"], value="DAs only", description="Show:"
)

# Shared collapse dicts/helpers from the Ma comparison script, loaded once.
_compare_mod, uber_categories, uber_palette = load_kamath_collapse_and_palette(repo_root)

_MARKERS_TO_PLOT = KAMATH_DA_MARKER_SYMBOLS

## 1 — All samples pooled: Ma (2025) agreement matrix/heatmaps from each reference build strategy

In [ ]:
_sort_by_dropdown_compare.observe(_redraw_all_vs_all_compare, names="value")
display(_sort_by_dropdown_compare)
display(widgets.HBox([_strategy_outs[n] for n in STRATEGIES]))
_redraw_all_vs_all_compare()

## 2 — Exploration of each sample deconvolution
In the dropdowns that will appear below, select: i) a sample and ii) either `All cell types` or `DAs only`

In [ ]:
display(sample_dropdown)
display(cell_type_view_dropdown)

In [ ]:
SAMPLE_NAME = sample_dropdown.value

# Normalize cell types across strategies and sources
_adata_by_strategy = {}
_cell_types_sel_by_strategy = {}
for n in STRATEGIES:
    h5ad_path = Path(BASE_INDIR, STRATEGY_DIR[n], SAMPLE_NAME + H5AD_SUFFIX)
    adata = ad.read_h5ad(h5ad_path)
    print(f"Strategy {n}: loaded {h5ad_path} {adata.shape}")

    cell_types = list(adata.obsm["q05_cell_abundance_w_sf"].columns)
    cell_types_ = [strip_q05_abundance_prefix(ct) for ct in cell_types]
    for ct, ct_ in zip(cell_types, cell_types_):
        adata.obs[ct_] = adata.obsm["q05_cell_abundance_w_sf"][ct].values
    _, cell_types_sel_ = selected_cell_types(cell_types, cell_types_, cell_type_view_dropdown.value, DA_UBER_CATEGORIES)

    _adata_by_strategy[n] = adata
    _cell_types_sel_by_strategy[n] = set(cell_types_sel_)

common_cell_types = sorted(set.intersection(*_cell_types_sel_by_strategy.values()))
for n in STRATEGIES:
    missing = _cell_types_sel_by_strategy[n] - set(common_cell_types)
    if missing:
        print(f"Strategy {n}: {len(missing)} cell type(s) not shared across all 3 "
              f"strategies, excluded from the row-aligned view: {sorted(missing)}")
print(f"{len(common_cell_types)} cell type(s) common to all 3 strategies: {common_cell_types}")

# Input from dropdown
cell_type_dropdown = dropdown(big=True, options=common_cell_types, description="Cell type:")

# Sliders
_PANEL_WIDTH_PX = 400
_vrange_widgets = {
    n: widgets.FloatRangeSlider(
        description=f"Strategy {n} (0 - 0):", readout=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width=f"{_PANEL_WIDTH_PX}px"),
    )
    for n in STRATEGIES
}


for n in STRATEGIES:
    _vrange_widgets[n].observe(functools.partial(_update_slider_label, n), names="value")

_DEFAULT_VMAX_PERCENTILE = 99.2  # Reset's default top of range, as a percentile of each strategy's values
_replot_button = widgets.Button(description="Replot")
_reset_button = widgets.Button(description=f"Reset (0% - {_DEFAULT_VMAX_PERCENTILE:g}%)")
_spatial_compare_out = widgets.Output()

cell_type_dropdown.observe(_reset_vmin_vmax_and_redraw, names="value")
_replot_button.on_click(_redraw_spatial_compare)
_reset_button.on_click(_reset_vmin_vmax_and_redraw)
_reset_vmin_vmax(cell_type_dropdown.value)

# Cell type dropdown
display(cell_type_dropdown)
display(widgets.HBox(
    [_vrange_widgets[n] for n in STRATEGIES],
    layout=widgets.Layout(width=f"{_PANEL_WIDTH_PX * len(STRATEGIES)}px", justify_content="space-between"),
))
display(widgets.HBox([_replot_button, _reset_button]))
display(_spatial_compare_out)

# Reset on very first draw. Needed to avoid image malforming.
asyncio.get_event_loop().call_later(0.5, _redraw_spatial_compare)

## 3 — Strategy N vs. Strategy M per-spot agreement scatter
Select X-axis and Y-axis strategies below (re-uses the `Sample:`/`Show:` dropdowns above), then re-run the next cell.

In [ ]:
_REFERENCE_OPTIONS = {f"Strategy {n}": n for n in STRATEGIES}

x_source_dropdown = dropdown(big=True, options=list(_REFERENCE_OPTIONS), description="X-axis:")
y_source_dropdown = dropdown(big=True, options=list(_REFERENCE_OPTIONS), description="Y-axis:", value="Strategy 2")
display(x_source_dropdown)
display(y_source_dropdown)

In [ ]:
x_source, y_source = x_source_dropdown.value, y_source_dropdown.value
x_abund = _load_reference_abundance(x_source, SAMPLE_NAME)
y_abund = _load_reference_abundance(y_source, SAMPLE_NAME)
assert list(x_abund.index) == list(y_abund.index), (
    f"Spot barcodes differ between '{x_source}' and '{y_source}' -- can't compare directly."
)

common_cell_types = sorted(set(x_abund.columns) & set(y_abund.columns))
if cell_type_view_dropdown.value == "DAs only":
    common_cell_types = [ct for ct in common_cell_types if uber_type_of(ct) in DA_UBER_CATEGORIES]
print(f"{len(common_cell_types)} cell type(s) comparable between '{x_source}' and '{y_source}': {common_cell_types}")

scatter_df = pd.concat([
    pd.DataFrame({
        "spot": x_abund.index, "cell_type": ct, "uber_category": uber_type_of(ct),
        "x": x_abund[ct].values, "y": y_abund[ct].values,
    })
    for ct in common_cell_types
], ignore_index=True)

# Each axis scaled to its own strategy's data range
x_axis_max = scatter_df["x"].max() * 1.05 if len(scatter_df) else 1
y_axis_max = scatter_df["y"].max() * 1.05 if len(scatter_df) else 1

# Overall best-fit line
slope, intercept = np.polyfit(scatter_df["x"], scatter_df["y"], 1) if len(scatter_df) else (1, 0)
fit_x = np.array([0, x_axis_max])
fit_y = slope * fit_x + intercept

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fit_x, y=fit_y, mode="lines",
    line=dict(dash="dash", color="gray"), name=f"best fit (y={slope:.2f}x+{intercept:.2f})", hoverinfo="skip",
))
for uber, group in scatter_df.groupby("uber_category"):
    r = group["x"].corr(group["y"])
    fig.add_trace(go.Scatter(
        x=group["x"], y=group["y"], mode="markers", name=f"{uber} (r={r:.2f})",
        marker=dict(size=5, color=_rgb_str(uber_palette.get(uber, (0, 0, 0))), opacity=0.6),
        customdata=group[["spot", "cell_type"]],
        hovertemplate="%{customdata[1]}<br>spot=%{customdata[0]}<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    ))

fig.update_layout(
    width=600, height=600,
    xaxis=dict(title=f"{x_source} abundance", range=[0, x_axis_max]),
    yaxis=dict(title=f"{y_source} abundance", range=[0, y_axis_max]),
    title=f"{SAMPLE_NAME}: {x_source} vs. {y_source} ({cell_type_view_dropdown.value})",
)
fig.show()